### CLIMATE FINANCE INTELLIGENCE

In [1]:
# Import necessary libs
import pandas as pd
import os
import matplotlib.pyplot as plt

In [2]:
# Load cleaned data
path = ("../data/processed/")

files = sorted([f for f in os.listdir(path) if f.endswith(".parquet")])

dfs = [pd.read_parquet(os.path.join(path, f)) for f in files]

df = pd.concat(dfs, ignore_index=True)

print(df.shape)

(3396556, 7)


In [3]:
print(df["Year"].min(), df["Year"].max())
print(df.isnull().sum().head())

2015 2024
Year              0
DonorName         0
RecipientName     0
SectorName        0
USD_Commitment    0
dtype: int64


In [4]:
# Extract Ethiopia Dataset
ethiopia_df = df[
    df["RecipientName"].str.contains("Ethiopia", case=False, na=False)
].copy()

print(ethiopia_df.shape)

(53372, 7)


In [5]:
# Climate Finance Growth Rate

trend = ethiopia_df.groupby("Year")["USD_Commitment"].sum()

growth_rate = trend.pct_change().mean()

print("Average Growth Rate: ", growth_rate)

Average Growth Rate:  0.08708072888263724


In [6]:
# Donor Dependency Index: formalizeing

donor_share = (
    ethiopia_df.groupby("DonorName")["USD_Commitment"].sum()
)

donor_share = donor_share / donor_share.sum()

dependency_index = (donor_share ** 2).sum()

print("Donor Dependency Index: ", dependency_index)

Donor Dependency Index:  0.17669722311570318


- closer to 0 → diversified
- closer to 1 → highly concentrated

In [7]:
# Sector Concentration Index

sector_share = (
    ethiopia_df.groupby("SectorName")["USD_Commitment"].sum()
)

sector_share = sector_share / sector_share.sum()

sector_index = (sector_share ** 2).sum()

print("Sector Concentration Index: ", sector_index)

Sector Concentration Index:  0.0672587281502394


In [8]:
# Climate Finance Efficiency Proxy
climate_tagged = ethiopia_df[
    (ethiopia_df["ClimateAdaptation"] > 0) | 
    (ethiopia_df["ClimateMitigation"] > 0)
]

effiiency_ratio = climate_tagged["USD_Commitment"].sum() / ethiopia_df["USD_Commitment"].sum()

print("Climate-targeted finance share: ", effiiency_ratio)

Climate-targeted finance share:  0.12821674714248557


In [9]:
# Carbon Market Reafiness Score
score = (
    (1 - dependency_index) * 0.3 +
    (1 - sector_index) * 0.2 + 
    growth_rate * 0.2 +
    effiiency_ratio * 0.3
) * 100

print("Carbon Market Reafiness Score: ", score)

Carbon Market Reafiness Score:  48.94202573545143


#### Ethiopia demonstrates moderate readiness for carbon market participation, supporeted by steady growth in climate finance and relatively diversified sectoral allocation. However, the low propertion of climate-targeted funding (approximately 13%) suggested that a significant share of financial flows is not directly aligned with climate mitigation and adaptation objectives, which may constrain effectice participation on carbon market mechanisms.

#### Breaking it down 

strengths:
   1. strong growth (8.7%):
       - Increasing investor/donor confidence
       - Expanding financial inflows
   2. Low Sector Concentration (0.067):
       - Good diversification
       - Not overly dependent on sector

Weakness:
   1. Low Climate-Targeted Finance (12%):
       - This is the biggest funding:
           - Meaning:
               - Most funding is NOT explicitly climate-focused
               - Weak alignment with:
                   - Mitigation
                   - Adaptation
                   - Carbon Markets
   2. Moderate Donor Coencentration (0.17)
       - Even though "Top 2 = 71%":
           - This confirms:
               - Still reliance on few donors
               - Vulnerability exists
               
Ethiopua is receiving funding, but not ehough of it is structured for carbon market participation.

#### While Ethiopia's climate finance landscpae show steady growth and diversified sectoral allocation, the relativity low share of explicity cimate-targeted funding highlights a structural gap in readiness for carbon market paritcipation, particularly under Article 6 mechanishms. Strengthening the alignment of financial flows with mitigation outcomes and improving limate tagging will be critical priorities ahead of COP32.

In [10]:
# Save Final Dataset
df.to_parquet("../data/processed/ethiopian_climate_analysis.parquet", index=False)

In [11]:
df==pd.read_parquet("../data/processed/ethiopian_climate_analysis.parquet")
df.head()

,Year,DonorName,RecipientName,SectorName,USD_Commitment,ClimateMitigation,ClimateAdaptation
0,2015,Belgium,Bosnia and Herzegovina,II.4. Banking & Financial Services,0.100356,NaN,NaN
1,2015,Belgium,Belarus,II.4. Banking & Financial Services,0.036339,NaN,NaN
2,2015,Belgium,Democratic Republic of the Congo,II.2. Communications,1.468142,NaN,NaN
3,2015,Belgium,Ghana,II.4. Banking & Financial Services,0.131247,NaN,NaN
4,2015,Belgium,Kenya,II.4. Banking & Financial Services,0.274794,NaN,NaN
